# Exploratory Data Analysis (EDA)
A análise exploratória de dados é a etapa inicial para entender o comportamento do evento-alvo, as distribuições das variáveis e as relações entre elas. O objetivo é:

- descobrir padrões, tendências e anomalias nos dados;
- verificar se há valores ausentes, outliers ou viés nas variáveis;
- selecionar variáveis relevantes para o modelo;
- descartar variáveis irrelevantes ou muito correlacionadas com o evento, evitando overfitting;
- preparar a base para um pipeline de modelagem mais robusto e confiável.

Esse trabalho ajuda a garantir que o modelo seja construído sobre informações consistentes e úteis, reduzindo erros e melhorando a capacidade de generalização.

In [1]:
# bibliotecas necessarias

import pandas as pd
import plotly.express as px
import datetime as dt

In [44]:
# configuracoes necessarias

## Leitura dos Dados

O projeto cont[em dois conjuntos de dados distintos que serão usados no desenvolvimento do sistema:

- um conjunto de dados com as informações temporais de acessos, pontuações e interações dos usuários com a plataforma 
    - `clientes.csv`
    - `produto.csv`
    - `transacoes_produto.csv`
    - `trnasacoes.csv`

- um conjunto de dados relativos a plataforma de educação e sua estrutura


In [45]:
transacoes = pd.read_csv('/home/sabrina/Documents/projetos/loyalty-predict/data/loyalty-system/transacoes.csv', sep=';')

In [46]:
print(transacoes.head(), '\\n')
print(transacoes.shape)
print(transacoes.info())

                            IdTransacao                             IdCliente  \
0  0000520a-a4e5-4977-b360-17be62fa0f2b  24782f0b-4683-4f35-976a-ea21d6714ba6   
1  000060e8-aa76-4286-a8d7-f30e6fa47edd  252a0923-3f79-45bb-b664-3040235c6c58   
2  000095de-3daa-4cfb-a0ae-e7b2c8bc3c9b  30f45a6d-ada5-4a17-8338-710e414eb6c6   
3  0000c010-a592-46f7-8b0b-6f841bee64ba  65662aff-44d6-4f06-b9d9-07445c6e5943   
4  0000dfbb-e14e-4ea7-a57f-60236869fffe  5f8fcbe0-6014-43f8-8b83-38cf2f4887b3   

                 DtCriacao  QtdePontos DescSistemaOrigem  
0  2025-09-17 12:28:41.864           1            twitch  
1  2024-07-23 12:49:49.874           1            twitch  
2  2025-09-17 13:38:57.479           1            twitch  
3  2025-10-03 12:23:39.779           1            twitch  
4  2024-02-20 13:21:45.613           1            twitch   \n
(345353, 5)
<class 'pandas.DataFrame'>
RangeIndex: 345353 entries, 0 to 345352
Data columns (total 5 columns):
 #   Column             Non-Null Count   Dtyp

In [47]:
# tratamento dos dados
transacoes['DtCriacao'] = pd.to_datetime(transacoes['DtCriacao'], format='%Y-%m-%d %H:%M:%S.%f').dt.normalize()

## Distribuição dos dados

In [48]:
# DAU - Daily Active Users
df_agp = transacoes.groupby('DtCriacao')['IdCliente'].nunique()

fig = px.line(df_agp, x=df_agp.index, y=df_agp.values, title='DAU - Daily Active Users', labels={'x': 'Data', 'y': 'Número de Usuários Ativos'})

fig.show()

In [49]:
# MAU - Monthly Active Users
df_agp = transacoes.groupby(pd.Grouper(key='DtCriacao', freq='ME'))['IdCliente'].nunique()

fig = px.line(df_agp, x=df_agp.index, y=df_agp.values, title='MAU - Monthly Active Users', labels={'x': 'Data', 'y': 'Número de Usuários Ativos'})

fig.show()

In [63]:
df_agp = transacoes.groupby('DtCriacao').agg(dau=('IdCliente', 'nunique'))
df_agp = df_agp.sort_index()
# df_agp['lag_1'] = df_agp['dau'].rolling(window=28).sum()
print(df_agp.head(n=20))

            dau
DtCriacao      
2024-01-27    1
2024-01-29   32
2024-01-30   46
2024-01-31   60
2024-02-01   61
2024-02-02   66
2024-02-05  233
2024-02-06  188
2024-02-07   84
2024-02-08   73
2024-02-09   72
2024-02-10    1
2024-02-11    1
2024-02-12   52
2024-02-13   40
2024-02-14   55
2024-02-15   70
2024-02-16  101
2024-02-17    1
2024-02-18    1


In [ ]:
# definicao do clico de vida do cliente
# recência: 7 - fiel, 8-15 - turista, 16-28 - desencantado, > 28 perdido (churn)
# idade base: primeiro acesso: curioso, era desencantado e virou fiel: reconquistado,
# era perdido e voltou: recuperado 

# importante para gestão da carteira de clientes e para o modelo de churn, 
# pois permite identificar clientes que estão em risco de churn e tomar medidas para retê-los.
# dados necessarios: idade na base, ultima interacao, penultima interacao

